# 10 - POLECAT vs GDELT Convergent Validity (2018-2024)

This notebook implements the bounded convergent-validity check scoped in the Section 4.7. It is not a re-run of the modelling pipeline. It tests one narrow question:

Over 2018-2024, do the two independently-produced event streams (GDELT and POLECAT) **agree** on the
bilateral-relations signal for our corridors - in event **volume**, in **cooperation-vs-conflict
balance**, and in **tone/intensity**?


In [ ]:
import pandas as pd, numpy as np, matplotlib as mpl
mpl.use("Agg")   # non-interactive backend so plt.show() never blocks under a headless kernel
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import glob, csv, re, warnings
warnings.filterwarnings("ignore")
mpl.rcParams.update({"figure.dpi":120,"savefig.dpi":200,"font.size":11,"axes.grid":True,
    "grid.alpha":0.25,"axes.spines.top":False,"axes.spines.right":False,"figure.autolayout":True})
ACCENT="#1F3864"; ORANGE="#C55A11"; TEAL="#2E7D6F"; RED="#B22222"
DATA=Path("data"); FIG=Path("eda_figures"); FIG.mkdir(exist_ok=True)
CACHE=DATA/"_polecat_cache"; CACHE.mkdir(parents=True, exist_ok=True)

# --- CONFIG ----------------------------------------------------------------------
POLECAT_DIR = Path("data/dataverse_files (1)")     # folder with ngecEvents.DV.YYYY.txt
OVERLAP_START, OVERLAP_END = "2018-01-01", "2024-12-31"
# ---------------------------------------------------------------------------------
print("Overlap:", OVERLAP_START, "->", OVERLAP_END, "| POLECAT dir:", POLECAT_DIR)

Overlap: 2018-01-01 -> 2024-12-31 | POLECAT dir: data\dataverse_files (1)


In [2]:
g=pd.read_parquet(DATA/"gdelt_features_monthly.parquet").reset_index()
g["month"]=pd.to_datetime(g["month"])
g=g[(g["month"]>=OVERLAP_START)&(g["month"]<=OVERLAP_END)].copy()
fam=["econ_coop","econ_ease","econ_demand","econ_reject","econ_threat","econ_coerce"]
assert (g["total_events"]==g[fam].sum(axis=1)).all(), "total_events is expected to equal the family sum"
COOP=["econ_coop","econ_ease"]; CONF=["econ_reject","econ_threat","econ_coerce"]
den=g["total_events"].replace(0,np.nan)
g["netcoop"]=((g[COOP].sum(axis=1)-g[CONF].sum(axis=1))/den).fillna(0.0)
gdelt=g[["pair_str","month","tier","total_events","netcoop","avg_tone"]].rename(
    columns={"total_events":"g_vol","netcoop":"g_netcoop","avg_tone":"g_tone"})
CORRIDORS=sorted(gdelt["pair_str"].unique())
print(f"GDELT overlap rows: {len(gdelt)} | corridors: {len(CORRIDORS)} | months: {gdelt['month'].nunique()}")
gdelt.head(3)

GDELT overlap rows: 1877 | corridors: 23 | months: 84


,pair_str,month,tier,g_vol,g_netcoop,g_tone
35,BRA-CHN,2018-01-01,T1,53.0,1.000000,0.268230
36,BRA-CHN,2018-02-01,T1,59.0,1.000000,-0.228753
37,BRA-CHN,2018-03-01,T1,121.0,0.983471,-1.074285


In [3]:
EU27={"AUSTRIA","BELGIUM","BULGARIA","CROATIA","CYPRUS","CZECHIA","CZECH REPUBLIC","DENMARK","ESTONIA",
 "FINLAND","FRANCE","GERMANY","GREECE","HUNGARY","IRELAND","ITALY","LATVIA","LITHUANIA","LUXEMBOURG",
 "MALTA","NETHERLANDS","POLAND","PORTUGAL","ROMANIA","SLOVAKIA","SLOVENIA","SPAIN","SWEDEN"}
ENTITY_ALIASES={
  "USA":{"USA","UNITED STATES","UNITED STATES OF AMERICA"},"CHN":{"CHINA","PEOPLE'S REPUBLIC OF CHINA"},
  "CAN":{"CANADA"},"BRA":{"BRAZIL"},"MEX":{"MEXICO"},"IND":{"INDIA"},
  "KOR":{"SOUTH KOREA","REPUBLIC OF KOREA","KOREA, REPUBLIC OF"},"JPN":{"JAPAN"},
  "EUR":EU27|{"EUROPEAN UNION"}}
entities=sorted(set(e for pr in CORRIDORS for e in pr.split("-")))
assert not [e for e in entities if e not in ENTITY_ALIASES], "add aliases"
N2E={}
for e,al in ENTITY_ALIASES.items():
    for a in al: N2E[a.upper().strip()]=e
CK={"|".join(sorted(pr.split("-"))):pr for pr in CORRIDORS}
NAMES_PAT=re.compile("|".join(sorted([re.escape(n) for n in N2E],key=len,reverse=True)),re.I)
def to_entities(cell):
    if pd.isna(cell): return set()
    return {N2E[x.strip().upper()] for x in str(cell).split(";") if x.strip().upper() in N2E}
def corridor_of(a,b):
    ea,eb=to_entities(a),to_entities(b)
    for x in ea:
        for y in eb:
            if x!=y:
                c=CK.get("|".join(sorted((x,y))))
                if c: return c
    return None
print("entities:", entities)
print("checks:", corridor_of("United States","China"), corridor_of("Germany","United States"),
      corridor_of("Israel; Greece; Cyprus","United States"))

entities: ['BRA', 'CAN', 'CHN', 'EUR', 'IND', 'JPN', 'KOR', 'MEX', 'USA']
checks: CHN-USA EUR-USA EUR-USA


In [4]:
F_DATE="Event Date"; F_ETYPE="Event Type"; F_CTX="Contexts"
F_ACTOR_CO="Actor Country"; F_RECIP_CO="Recipient Country"
F_INTENSITY="Event Intensity"; F_QUAD="Quad Code"
USE=[F_DATE,F_ETYPE,F_QUAD,F_CTX,F_ACTOR_CO,F_RECIP_CO,F_INTENSITY]
RAW=sorted(glob.glob(str(POLECAT_DIR/"*.txt")))+sorted(glob.glob(str(POLECAT_DIR/"*.txt.gz")))
CACHED=sorted(glob.glob(str(CACHE/"agg_*.parquet")))
POLECAT_PRESENT = bool(RAW or CACHED)
if RAW:
    s=pd.read_csv(RAW[0],sep="\t",dtype=str,quoting=csv.QUOTE_NONE,on_bad_lines="skip",
                  usecols=lambda c:c in USE, nrows=200_000)   # sample for display only
    s=s[s[F_ETYPE]!=F_ETYPE]
    print("taxonomy sample from", Path(RAW[0]).name, "(first 200k rows) | rows", len(s))
    print("\nEvent Type:", sorted(s[F_ETYPE].dropna().unique().tolist()))
    print("\nQuad Code:", s[F_QUAD].value_counts(dropna=False).to_dict())
    ctx=s[F_CTX].dropna().str.split("|").explode().str.strip().str.lower()   # vectorised flatten
    print("\nContexts (NO economic/trade tag exists):", ctx.value_counts().to_dict())
    print("\nActor Country sample:", s[F_ACTOR_CO].dropna().str.strip().value_counts().head(10).index.tolist())
elif CACHED:
    print("Raw files absent; using per-year cache for aggregation (taxonomy display skipped).")
else:
    print("POLECAT not found in", POLECAT_DIR.resolve())

taxonomy sample from ngecEvents.DV.2018.txt (first 200k rows) | rows 200000

Event Type: ['ACCUSE', 'AID', 'ASSAULT', 'COERCE', 'CONCEDE', 'CONSULT', 'COOPERATE', 'MOBILIZE', 'PROTEST', 'REJECT', 'REQUEST', 'RETREAT', 'SANCTION', 'THREATEN']

Quad Code: {'VERBAL CONFLICT': 87328, 'MATERIAL CONFLICT': 67639, 'MATERIAL COOPERATION': 37697, 'VERBAL COOPERATION': 7336}

Contexts (NO economic/trade tag exists): {'military': 43529, 'diplomatic': 32917, 'election': 26700, 'terrorism': 16750, 'migration': 11089, 'human_rights': 9925, 'intelligence': 6535, 'religion_ethnicity': 4204, 'disasters': 3129, 'health': 2207, 'environment': 2204, 'cyber': 2202, 'illegal_drugs': 1711, 'human_security': 1140, 'peacekeeping': 1096, 'lgbt': 739, 'asylum': 648}

Actor Country sample: ['India', 'Russia', 'United States', 'United Kingdom', 'Israel', 'China', 'Turkey', 'Saudi Arabia', 'None; None', 'Iran']


In [5]:
def process_year(fp):
    d=pd.read_csv(fp,sep="\t",dtype=str,quoting=csv.QUOTE_NONE,on_bad_lines="skip",usecols=lambda c:c in USE)
    d=d[d[F_ETYPE]!=F_ETYPE]
    d=d[d[F_ACTOR_CO].str.contains(NAMES_PAT,na=False)&d[F_RECIP_CO].str.contains(NAMES_PAT,na=False)]
    d[F_DATE]=pd.to_datetime(d[F_DATE],errors="coerce"); d=d.dropna(subset=[F_DATE])
    d=d[(d[F_DATE]>=OVERLAP_START)&(d[F_DATE]<=OVERLAP_END)]
    d["month"]=d[F_DATE].values.astype("datetime64[M]")
    qc=d[F_QUAD].fillna("").str.upper()
    d["valence"]=np.select([qc.str.contains("COOPERATION"),qc.str.contains("CONFLICT")],["coop","conflict"],default="neutral")
    ea=d[F_ACTOR_CO].map(to_entities); eb=d[F_RECIP_CO].map(to_entities)
    k=(ea.str.len()>0)&(eb.str.len()>0); d=d[k]; ea=ea[k].values; eb=eb[k].values
    def match(a,b):
        for x in a:
            for y in b:
                if x!=y:
                    c=CK.get("|".join(sorted((x,y))))
                    if c: return c
        return None
    d["pair_str"]=[match(a,b) for a,b in zip(ea,eb)]
    d=d.dropna(subset=["pair_str"]); d["intensity"]=pd.to_numeric(d[F_INTENSITY],errors="coerce")
    return d.groupby(["pair_str","month"]).agg(p_vol=("valence","size"),
        p_coop=("valence",lambda s:(s=="coop").sum()),p_conf=("valence",lambda s:(s=="conflict").sum()),
        p_intensity=("intensity","mean")).reset_index()

def get_polecat_monthly():
    parts=[]
    for fp in RAW:
        yr="".join(ch for ch in Path(fp).stem if ch.isdigit())[-4:]
        cf=CACHE/f"agg_{yr}.parquet"
        if cf.exists(): parts.append(pd.read_parquet(cf))
        else:
            a=process_year(fp); a.to_parquet(cf); parts.append(a); print("  built cache for", yr)
    if not parts and CACHED:
        parts=[pd.read_parquet(f) for f in CACHED]
    if not parts: return None
    allp=pd.concat(parts,ignore_index=True)
    out=allp.groupby(["pair_str","month"]).agg(p_vol=("p_vol","sum"),p_coop=("p_coop","sum"),
        p_conf=("p_conf","sum"),p_intensity=("p_intensity","mean")).reset_index()
    den=out["p_vol"].replace(0,np.nan); out["p_netcoop"]=((out["p_coop"]-out["p_conf"])/den).fillna(0.0)
    return out

polm=get_polecat_monthly() if POLECAT_PRESENT else None
if polm is not None:
    print(f"POLECAT corridor-months: {len(polm)} | corridors: {polm['pair_str'].nunique()} "
          f"| tagged events: {int(polm['p_vol'].sum()):,}")
    display(polm.head(3))
else:
    print("POLECAT absent.")

POLECAT corridor-months: 1696 | corridors: 23 | tagged events: 59,544


,pair_str,month,p_vol,p_coop,p_conf,p_intensity,p_netcoop
0,BRA-CHN,2018-03-01,1,0,1,-3.0,-1.0
1,BRA-CHN,2018-05-01,1,1,0,10.0,1.0
2,BRA-CHN,2018-07-01,4,2,2,5.0,0.0


Convergent-validity metrics

Spearman is primary (rank-based, robust to the heavy right-skew of counts). Three views: **pooled**;
**within-corridor** (demeaned by corridor - a Simpson guard removing cross-corridor level effects); and the
**per-corridor** distribution. A lead-lag check confirms whether agreement is contemporaneous.

In [6]:
def cb(a,b,label):
    a=np.asarray(a,float); b=np.asarray(b,float); ok=np.isfinite(a)&np.isfinite(b)
    if ok.sum()<5: return dict(signal=label,n=int(ok.sum()),pearson=np.nan,spearman=np.nan)
    return dict(signal=label,n=int(ok.sum()),pearson=round(pearsonr(a[ok],b[ok])[0],3),
                spearman=round(spearmanr(a[ok],b[ok])[0],3))
if polm is not None and len(polm):
    m=gdelt.merge(polm,on=["pair_str","month"],how="inner")
    print(f"matched corridor-months: {len(m)} (corridors={m['pair_str'].nunique()})")
    m["gv"]=np.log1p(m["g_vol"]); m["pv"]=np.log1p(m["p_vol"])
    pooled=pd.DataFrame([
        cb(m["gv"],m["pv"],"volume (GDELT trade vs POLECAT all)"),
        cb(m["g_netcoop"],m["p_netcoop"],"coop-conflict balance"),
        cb(m["g_tone"],m["p_intensity"],"tone vs intensity")])
    def dm(s): return s-s.groupby(m["pair_str"]).transform("mean")
    within=pd.DataFrame([
        cb(dm(m["gv"]),dm(m["pv"]),"volume (within)"),
        cb(dm(m["g_netcoop"]),dm(m["p_netcoop"]),"balance (within)"),
        cb(dm(m["g_tone"]),dm(m["p_intensity"]),"tone (within)")])
    print("\nPOOLED:"); display(pooled)
    print("WITHIN-CORRIDOR:"); display(within)
    per=[]
    for c,gc in m.groupby("pair_str"):
        if len(gc)>=6:
            per.append({"pair_str":c,"n":len(gc),
                "rho_vol":round(spearmanr(gc["gv"],gc["pv"])[0],2),
                "rho_netcoop":round(spearmanr(gc["g_netcoop"],gc["p_netcoop"])[0],2),
                "rho_tone":round(spearmanr(gc["g_tone"],gc["p_intensity"])[0],2)})
    per=pd.DataFrame(per).sort_values("rho_vol",ascending=False)
    print(f"\nPER-CORRIDOR (>=6 mo) - median rho_vol={per['rho_vol'].median():.2f}:"); display(per)
    pooled.assign(view="pooled").to_csv(DATA/"polecat_convergence_pooled.csv",index=False)
    within.assign(view="within").to_csv(DATA/"polecat_convergence_within.csv",index=False)
    per.to_csv(DATA/"polecat_convergence_percorridor.csv",index=False)
    mm=m.sort_values(["pair_str","month"])
    print("\nLEAD-LAG (GDELT vol vs POLECAT vol shifted k months, pooled Spearman):")
    for k in [-2,-1,0,1,2]:
        print(f"  k={k:+d}: rho={cb(mm['gv'],mm.groupby('pair_str')['pv'].shift(k),'')['spearman']}")
else:
    m=None; print("comparison skipped (POLECAT absent).")

matched corridor-months: 1663 (corridors=23)

POOLED:


,signal,n,pearson,spearman
0,volume (GDELT trade vs POLECAT all),1663,0.547,0.454
1,coop-conflict balance,1663,0.148,0.153
2,tone vs intensity,1663,0.123,0.159


WITHIN-CORRIDOR:


,signal,n,pearson,spearman
0,volume (within),1663,0.284,0.247
1,balance (within),1663,0.063,0.065
2,tone (within),1663,0.071,0.104



PER-CORRIDOR (>=6 mo) - median rho_vol=0.25:


,pair_str,n,rho_vol,rho_netcoop,rho_tone
17,EUR-USA,78,0.68,-0.18,0.19
12,CHN-USA,78,0.62,0.32,0.57
6,CAN-USA,78,0.55,0.11,0.08
20,JPN-USA,78,0.48,0.03,0.28
22,MEX-USA,78,0.47,-0.07,0.01
7,CHN-EUR,78,0.41,0.26,0.21
19,JPN-KOR,78,0.39,0.14,0.32
21,KOR-USA,78,0.39,0.06,0.21
5,CAN-EUR,78,0.36,-0.17,-0.20
18,IND-USA,78,0.31,-0.03,-0.02



LEAD-LAG (GDELT vol vs POLECAT vol shifted k months, pooled Spearman):
  k=-2: rho=0.418
  k=-1: rho=0.428
  k=+0: rho=0.454
  k=+1: rho=0.427
  k=+2: rho=0.42


In [ ]:
if m is not None and len(m):
    fig,ax=plt.subplots(1,3,figsize=(15,4.3))
    ax[0].scatter(m["gv"],m["pv"],s=10,alpha=.35,color=ACCENT)
    ax[0].set(xlabel="GDELT log trade-event volume",ylabel="POLECAT log all-event volume",title="A. Event volume")
    for c,col in [("EUR-USA",ACCENT),("CHN-USA",ORANGE)]:
        s=m[m["pair_str"]==c].sort_values("month")
        if len(s):
            ax[1].plot(s["month"],s["g_netcoop"],color=col,lw=1.6,label=f"{c} GDELT")
            ax[1].plot(s["month"],s["p_netcoop"],color=col,lw=1.2,ls="--",label=f"{c} POLECAT")
    ax[1].axhline(0,color="grey",lw=.7); ax[1].set(title="B. Coop-conflict balance",ylabel="net_coop"); ax[1].legend(fontsize=7)
    per=pd.read_csv(DATA/"polecat_convergence_percorridor.csv")
    cols=[TEAL if v>=0 else RED for v in per["rho_vol"]]
    ax[2].barh(per["pair_str"],per["rho_vol"],color=cols); ax[2].axvline(0,color="grey",lw=.7)
    ax[2].set(title="C. Per-corridor Spearman (volume)",xlabel="rho"); ax[2].invert_yaxis()
    fig.suptitle("GDELT vs POLECAT convergent validity, 2018-2024",fontsize=12,y=1.02)
    fig.savefig(FIG/"polecat_convergence.png",bbox_inches="tight",dpi=200); plt.show()
    print("saved eda_figures/polecat_convergence.png")
else:
    print("no figure (POLECAT absent).")

saved eda_figures/polecat_convergence.png


In [8]:
assert abs(spearmanr(np.log1p(gdelt['g_vol']),np.log1p(gdelt['g_vol']))[0]-1.0)<1e-9
print("[PASS] GDELT self Spearman = 1.0")
assert corridor_of("United States","China")==corridor_of("China","United States")
assert corridor_of("United States","United States") is None
assert corridor_of("Israel; Greece; Cyprus","United States")=="EUR-USA"
print("[PASS] tagger undirected, rejects self-pairs, handles multi-country names")
_qc=pd.Series(["VERBAL COOPERATION","MATERIAL CONFLICT","",np.nan]).fillna("").str.upper()
_v=np.select([_qc.str.contains("COOPERATION"),_qc.str.contains("CONFLICT")],["coop","conflict"],default="neutral")
assert list(_v)==["coop","conflict","neutral","neutral"]
print("[PASS] Quad Code string -> valence"); print("Notebook OK.")

[PASS] GDELT self Spearman = 1.0
[PASS] tagger undirected, rejects self-pairs, handles multi-country names
[PASS] Quad Code string -> valence
Notebook OK.
